# 01 - OpenSky Data Ingestion
Collect ADS-B state vectors based on configuration.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.append(os.path.abspath('..'))
import time
from src.utils import load_config
from src.opensky_client import OpenSkyClient

In [2]:
config = load_config('../configs/config.yaml')
client = OpenSkyClient(
    polling_interval=config['opensky']['polling_interval_seconds'],
    client_id=config['opensky'].get('clientId'),
    client_secret=config['opensky'].get('clientSecret'),
    bbox=config['opensky'].get('bbox')
)

INFO:src.opensky_client:Successfully fetched OpenSky access token.


In [3]:
end_time = int(time.time())
start_time = end_time - (config['opensky']['collection_hours'] * 3600)
print(f'Fetching from {start_time} to {end_time} ({config["opensky"]["collection_hours"]} hours)')
print(f'Expected queries: {(end_time - start_time) // config["opensky"]["polling_interval_seconds"]}')

Fetching from 1773247348 to 1773250948 (1 hours)
Expected queries: 120


In [4]:
# Note: OpenSky allows 400 requests/day for anonymous users and 4000/day for authenticated users.
# Please configure `clientId` and `clientSecret` in `config.yaml` to avoid 403 API Rate Limit errors.
df = client.fetch_states_range(start_time, end_time)
date_str = time.strftime('%Y%m%d', time.localtime(end_time))
client.save_raw_states(df, f"../{config['paths']['raw_data_dir']}", date_str)

INFO:src.opensky_client:Fetching states from 1773247348 to 1773250948 (120 requests)
Fetching OpenSky Data: 100%|██████████| 120/120 [04:30<00:00,  2.25s/it]
INFO:src.opensky_client:Saving 103344 records to ..\data\raw\states_20260311.parquet
